# Skillmodels Quickstart

This tutorial demonstrates the basic workflow for estimating a latent factor model
using skillmodels. We'll use Example 2 from the Cunha, Heckman, and Schennach (2010)
replication files.

In [ ]:
import pandas as pd

from skillmodels.chs import get_maximization_inputs
from skillmodels.common.config import TEST_DATA_DIR
from skillmodels.test_data.model2 import MODEL2

<![CDATA[## Loading Model Specification and Data

Models are specified as Python dataclasses. See the
[Model Specifications](../how_to_guides/model_specs.md) guide for details.

For this tutorial, we use a model specification that ships with skillmodels.]]>

In [ ]:
model = MODEL2

# Show the structure
print("Factors:", list(model.factors.keys()))

In [ ]:
data = pd.read_stata(TEST_DATA_DIR / "model2_simulated_data.dta")
data = data.set_index(["caseid", "period"])
data.head()

## Getting Maximization Inputs

The main entry point is `get_maximization_inputs()`. It takes a model specification
and dataset, and returns everything needed to maximize the likelihood using optimagic:

- `loglike`: The compiled log-likelihood function
- `gradient`: The gradient of the log-likelihood
- `loglike_and_gradient`: Combined function (more efficient)
- `debug_loglike`: Uncompiled version for debugging
- `params_template`: Parameter DataFrame with bounds and starting values
- `constraints`: Parameter constraints for optimization

In [ ]:
max_inputs = get_maximization_inputs(model, data)
print("Available keys:", list(max_inputs.keys()))

## Parameter Template

The `params_template` is a pandas DataFrame with:
- A MultiIndex identifying each parameter (category, period, name1, name2)
- Columns for `value` (to be filled with starting values), `lower_bound`, `upper_bound`

In [ ]:
params_template = max_inputs["params_template"]
params_template.head(10)

## Choosing Starting Values

Good starting values are important for optimization. As a rule of thumb:

- If measurements are standardized, use 1.0 for free loadings and 0.0 for free intercepts
- Start measurement and shock standard deviations slightly larger than expected
- Initial state means can often start at 0

Here we set reasonable defaults:

In [ ]:
params = params_template.copy()

# Set starting values by category
for category in params.index.get_level_values("category").unique():
    if category == "loadings":
        params.loc[category, "value"] = 1.0
    elif category == "controls":
        params.loc[category, "value"] = 0.0
    elif category in ("meas_sds", "shock_sds") or category == "initial_cholcovs":
        params.loc[category, "value"] = 0.5
    elif category == "initial_states":
        params.loc[category, "value"] = 0.0
    elif category == "mixture_weights":
        params.loc[category, "value"] = 1.0
    elif category == "transition":
        # Set transition parameters to reasonable defaults
        params.loc[category, "value"] = 0.5

params.head(10)

## JAX Compilation

Skillmodels uses JAX for just-in-time compilation and automatic differentiation.
The first call to `loglike` or `gradient` triggers compilation, which takes a few
seconds. Subsequent calls are very fast.

In [ ]:
loglike = max_inputs["loglike"]
gradient = max_inputs["gradient"]
loglike_and_gradient = max_inputs["loglike_and_gradient"]

In [ ]:
# First call includes compilation time
loglike_value = loglike(params)
print(f"Log-likelihood at starting values: {loglike_value:.2f}")

## Constraints

Skillmodels automatically generates constraints from the model specification:
- Fixed parameters (normalized loadings and intercepts)
- Stagemap equality constraints
- Bound constraints

You can add additional constraints for your specific model.

In [ ]:
constraints = max_inputs["constraints"]
print(f"Number of auto-generated constraints: {len(constraints)}")

## Estimation with optimagic

To estimate the model, use optimagic's `maximize` function:

```python
import optimagic as om

result = om.maximize(
    fun=loglike,
    params=params,
    algorithm="scipy_lbfgsb",
    fun_and_jac=loglike_and_gradient,
    constraints=constraints,
)
```

The `fun_and_jac` argument is important: it uses the combined function that
computes both the likelihood and gradient efficiently.

## Next Steps

- See the [Model Specifications](../how_to_guides/model_specs.md) guide for details
  on writing model specifications
- See the [Simulation](../how_to_guides/how_to_simulate_dataset.ipynb) guide for
  generating synthetic data
- After estimation, use `get_filtered_states()` to extract latent factor estimates